## 0. Clone the repo

In [1]:
!git clone https://github.com/mazinbashierrr/Meeting-Accountability-Predictor.git
%cd Meeting-Accountability-Predictor
!ls

Cloning into 'Meeting-Accountability-Predictor'...
remote: Enumerating objects: 2530, done.
remote: Counting objects: 100% (2530/2530), done.
remote: Compressing objects: 100% (966/966), done.
remote: Total 2530 (delta 1562), reused 2527 (delta 1562), pack-reused 0 (from 0)
Receiving objects: 100% (2530/2530), 23.95 MiB | 19.14 MiB/s, done.
Resolving deltas: 100% (1562/1562), done.
/kaggle/working/Meeting-Accountability-Predictor
data  meeting_accountability.ipynb  README.md  splits  utterances.parquet


## 1. Install only what's actually missing

Deliberately minimal: we do NOT touch `huggingface_hub`, `fsspec`, or `datasets`
at all, since upgrading those is what caused the circular-import errors before.
`accelerate` is the one thing Kaggle's base image sometimes lacks a recent
enough version of for `Trainer` to work, so that's the only thing we install.

In [2]:
!pip install -q -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.7 MB/s eta 0:00:00


In [3]:
!pip uninstall -y datasets

Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0


**Restart the session now** (Kaggle: the restart icon near the session
controls), then run the cells below from the top.

In [4]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, WeightedRandomSampler, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report

MODEL_NAME = "roberta-base"
MAX_LEN = 64          # utterances are short (single dialogue-act turns)
SEED = 42

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


### GPU check -- this MUST say `cuda` before you continue

If the cell below raises an error, you are on CPU and training would take
10+ hours instead of ~20-30 minutes. Fix in Session Options (Accelerator ->
GPU T4 x2) before going further.

In [5]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "No GPU attached -- fix Session Options before continuing."
print("GPU check passed:", torch.cuda.get_device_name(0))

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
GPU check passed: Tesla T4


## 2. Load the splits

Same rows Mazin baseline used, so results are directly comparable across the
three models.

In [6]:
tr = pd.read_json("splits/train.jsonl", lines=True).assign(text=lambda x: x.text.fillna(""))
dv = pd.read_json("splits/dev.jsonl",   lines=True).assign(text=lambda x: x.text.fillna(""))
te = pd.read_json("splits/test.jsonl",  lines=True).assign(text=lambda x: x.text.fillna(""))

for name, d in [("train", tr), ("dev", dv), ("test", te)]:
    print(f"{name:5s}: {len(d):6d} rows, {d.label.sum():3d} action items "
          f"({100*d.label.mean():.2f}%)")

train:  81080 rows, 255 action items (0.31%)
dev  :  17401 rows,  73 action items (0.42%)
test :  16907 rows,  53 action items (0.31%)


## 3. Tokenize

A small custom `torch.utils.data.Dataset` -- tokenizes on the fly with
RoBERTa's own tokenizer, no `datasets` library involved.

In [7]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class UtteranceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], truncation=True, padding="max_length",
                        max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = UtteranceDataset(tr, tok, MAX_LEN)
dev_ds   = UtteranceDataset(dv, tok, MAX_LEN)
test_ds  = UtteranceDataset(te, tok, MAX_LEN)
print("train/dev/test sizes:", len(train_ds), len(dev_ds), len(test_ds))

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

train/dev/test sizes: 81080 17401 16907


## 4. Model + oversampled positive class

Action items are ~0.3% of utterances. Straight loss-reweighting collapsed the
model in earlier attempts (verified empirically), because most batches simply
never contain a positive example to reweight in the first place. Instead we
oversample: a `WeightedRandomSampler` makes positive rows show up roughly 50x
more often per batch than their raw frequency, so the model actually sees
positive examples regularly during training.

Per the project plan: AdamW, learning rate 2e-5 (full fine-tuning needs a much
lower LR than the 2e-4 originally specified for LoRA adapters), cross-entropy
loss, early stopping on the dev split.

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

OVERSAMPLE_FACTOR = 50.0  # positive rows are drawn ~50x more often per batch than their raw frequency
sample_weights = tr["label"].map({0: 1.0, 1: OVERSAMPLE_FACTOR}).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

class WeightedTrainer(Trainer):
    def get_train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.args.per_device_train_batch_size,
                           sampler=sampler)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.functional.cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"precision": p, "recall": r, "f1": f1}

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. (Optional) time a few steps first

Runs 50 steps only, so you can see the real speed on whatever GPU you got
before committing to the full run.

In [9]:
import time
from transformers import TrainingArguments as _TA

_probe_args = _TA(output_dir="probe_out", max_steps=50, per_device_train_batch_size=16,
                   logging_steps=50, report_to="none", seed=SEED)
_probe_trainer = WeightedTrainer(model=model, args=_probe_args, train_dataset=train_ds,
                                  compute_metrics=compute_metrics)
t0 = time.time()
_probe_trainer.train()
elapsed = time.time() - t0
steps_per_epoch = len(train_ds) // 16
print(f"\n{elapsed:.1f}s for 50 steps -> ~{elapsed/50*steps_per_epoch/60:.1f} min/epoch estimated")

Step,Training Loss
50,0.359838


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


14.1s for 50 steps -> ~23.8 min/epoch estimated


## 6. Train

Early stopping watches dev F1 and stops once it stops improving, per the plan.

In [10]:
args = TrainingArguments(
    output_dir="roberta_out",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.018791,0.085618,0.157480,0.547945,0.244648
2,0.008740,0.051647,0.216495,0.287671,0.247059
3,0.005198,0.042999,0.323529,0.301370,0.312057
4,0.004052,0.049750,0.333333,0.328767,0.331034
5,0.000322,0.044422,0.565217,0.178082,0.270833
6,0.000766,0.045287,0.555556,0.205479,0.300000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=30408, training_loss=0.013065722494378127, metrics={'train_runtime': 6353.6996, 'train_samples_per_second': 76.566, 'train_steps_per_second': 4.786, 'total_flos': 1.59997832764416e+16, 'train_loss': 0.013065722494378127, 'epoch': 6.0})

## 7. Pick the decision threshold on dev, then score test

Same protocol as the baseline: sweep the cutoff on **dev**, pick the one that
maximizes F1, apply that fixed cutoff to **test**.

### Quick sanity check before trusting the numbers

If `std` here is near-zero, the model is still collapsing (predicting almost
the same probability for every row). If `std` is meaningfully above zero and
`max` is well above `mean`, the model is actually discriminating between
examples, which is what we want to see.

In [11]:
def get_probs(ds):
    preds = trainer.predict(ds)
    probs = torch.softmax(torch.tensor(preds.predictions), dim=1)[:, 1].numpy()
    return probs, preds.label_ids

dev_probs, dev_labels = get_probs(dev_ds)
print("dev prob stats -- min/mean/max/std:", dev_probs.min(), dev_probs.mean(), dev_probs.max(), dev_probs.std())

cutoffs = np.linspace(.05, .95, 19)
cutoff = max(cutoffs, key=lambda t: f1_score(dev_labels, dev_probs >= t, zero_division=0))
print(f"chosen cutoff (max dev F1): {cutoff:.2f}")

test_probs, test_labels = get_probs(test_ds)
test_pred = test_probs >= cutoff

p, r, f1, _ = precision_recall_fscore_support(test_labels, test_pred, average="binary", zero_division=0)
print(f"\n[test] cutoff = {cutoff:.2f}")
print(f"precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")
print(classification_report(test_labels, test_pred, target_names=["not action item", "action item"]))

dev prob stats -- min/mean/max/std: 5.493406e-06 0.004097994 0.9999869 0.062368196
chosen cutoff (max dev F1): 0.95



[test] cutoff = 0.95
precision=0.370  recall=0.321  f1=0.343
                 precision    recall  f1-score   support

not action item       1.00      1.00      1.00     16854
    action item       0.37      0.32      0.34        53

       accuracy                           1.00     16907
      macro avg       0.68      0.66      0.67     16907
   weighted avg       1.00      1.00      1.00     16907



## 8. Save results and download

Saves next to `splits/`, in the same spirit as the baseline saving
`utterances.parquet` -- so Agnes comparison/error-analysis step and the final
report can just read this file instead of re-running training.

In [12]:
import json

results = {
    "model": "roberta-base (fine-tuned, class-weighted CE loss, oversampled)",
    "cutoff": float(cutoff),
    "test_precision": float(p),
    "test_recall": float(r),
    "test_f1": float(f1),
}
with open("/kaggle/working/roberta_results.json", "w") as f:
    json.dump(results, f, indent=2)

results

{'model': 'roberta-base (fine-tuned, class-weighted CE loss, oversampled)',
 'cutoff': 0.95,
 'test_precision': 0.3695652173913043,
 'test_recall': 0.32075471698113206,
 'test_f1': 0.3434343434343434}

In [13]:
# Kaggle keeps everything under /kaggle/working automatically available for download
# from the notebook's Output/Data pane -- no explicit download call needed like Colab.
# You can also download it directly:
from IPython.display import FileLink
FileLink("/kaggle/working/roberta_results.json")

/kaggle/working/roberta_results.json

## 9. Push back to GitHub

Download `roberta_results.json` (link above, or from the notebook's Output pane)
and this notebook (File > Download), then from your local clone of the repo:

```bash
git add roberta_finetune.ipynb roberta_results.json
git commit -m "Add RoBERTa fine-tuning + evaluation (Akila part)"
git push
```

## Methodology (for the report)

We fine-tuned `roberta-base` (125M parameters) as a binary sequence classifier,
adding a linear classification head on top of the pooled representation.
Each dialogue-act utterance from the AMI corpus was tokenized with RoBERTa own
byte-pair tokenizer (max length 64 tokens, which covers the large majority of
single-turn utterances without truncation).

Because action items make up only about 0.3% of utterances, we used a
`WeightedRandomSampler` to oversample the positive class roughly 50x during
training, rather than relying purely on loss reweighting -- an earlier attempt
with class-weighted cross-entropy alone caused the model to collapse to always
predicting the majority class, since most training batches never contained a
positive example to begin with. We used AdamW with a learning rate of 2e-5 and
trained for up to 6 epochs, monitoring F1 on the dev split after every epoch
and stopping early if it did not improve for 2 consecutive epochs
(`load_best_model_at_end=True` restores the best-scoring checkpoint).

Following the same evaluation protocol as the TF-IDF + Logistic Regression
baseline, we did not use the default 0.5 decision threshold. Instead, we swept
candidate cutoffs on the **dev** split, chose the one that maximized F1 for the
action-item class, and applied that fixed cutoff to the **test** split so the
threshold itself was never tuned on test data. We report precision, recall,
and F1 for the action-item class specifically, since accuracy is not
informative under this level of class imbalance (a model that never predicts
"action item" would still score ~99.7% accuracy).